# บทที่ 0 พื้นฐานการเขียนโปรแกรมภาษาไพทอนสำหรับนักฟิสิกส์
โน้ตบุ๊กประกอบตำรา *วิทยาศาสตร์ข้อมูลและการเรียนรู้ของเครื่องสำหรับนักฟิสิกส์*
โค้ดทุกเซลล์คัดลอกมาจากสคริปต์ในโฟลเดอร์ `ch00_python_basics/` หัวเซลล์ **โค้ดที่ N.M** ตรงกับเลขในตำรา

**วิธีใช้**
1. รันเซลล์ **เตรียมโค้ด** ด้านล่างก่อนเสมอ (ทุกครั้งที่เปิดโน้ตบุ๊กหรือเริ่ม session ใหม่)
2. แต่ละหัวข้อเริ่มด้วยเซลล์ที่ขึ้นต้นด้วย `%reset -f` ซึ่งล้างตัวแปรเก่า ทำให้ผลเหมือนรันสคริปต์นั้นเดี่ยว ๆ
   ให้รันเซลล์ในหัวข้อเดียวกันเรียงจากบนลงล่าง (Shift+Enter) หรือใช้ Runtime → Run all
3. ใช้หน่วยประมวลผลกลาง (CPU) ตามค่าเริ่มต้น ไม่ต้องเปลี่ยนเป็น GPU
4. ไฟล์ที่สร้างขึ้นอยู่ใน `ch00_python_basics/outputs/` และหายเมื่อ session จบ ถ้าต้องการเก็บ ให้คัดลอกไปยัง Google Drive

ตัวเลขส่วนใหญ่ควรตรงกับตำรา การคำนวณที่ไวต่อการปัดเศษ (การฝึกโครงข่ายประสาทเทียม
และมอนติคาร์โลในพิกัดต่อเนื่อง) อาจต่างเล็กน้อยเมื่อรันบนเครื่องอื่น ดู `ch00_python_basics/README.md`


In [ ]:
# เตรียมโค้ด: ดึงโค้ดจาก GitHub (เมื่อรันบน Colab) และติดตั้งเฉพาะไลบรารีที่ยังไม่มี
import os, sys, subprocess, importlib.util
CHAPTER = 'ch00_python_basics'
REPO = 'https://github.com/busarapa-stack/dsml-for-physicists.git'
if 'google.colab' in sys.modules:
    ROOT = '/content/dsml-for-physicists'
    if not os.path.isdir(ROOT):
        subprocess.run(['git', 'clone', '-q', REPO, ROOT], check=True)
else:                                   # Jupyter บนเครื่องตนเอง: เปิดจากโฟลเดอร์ notebooks/
    ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(os.path.join(ROOT, CHAPTER))
os.makedirs('data', exist_ok=True); os.makedirs('outputs', exist_ok=True)
NEED = {'matplotlib': 'matplotlib', 'numpy': 'numpy'}
missing = [req for mod, req in NEED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
print('โฟลเดอร์ทำงาน:', os.getcwd())
print('ติดตั้งเพิ่ม:', ', '.join(missing) if missing else 'ไม่มี')


## `01_make_pendulum_data.py`

Ch.0  Create the SYNTHETIC pendulum data set used throughout Chapter 0.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 01_make_pendulum_data.py  จากโฟลเดอร์ของบท
__file__ = '01_make_pendulum_data.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.0  Create the SYNTHETIC pendulum data set used throughout Chapter 0.

Book references: sec:py-numpy (the file is read in 06_numpy_matplotlib.py), exercise E0.7
Five string lengths, ten trials each. Each trial is the time for ten complete
swings measured with a hand-held stopwatch, rounded to 0.01 s, as in a first-year
physics laboratory. The "true" value used to generate the data is g = 9.78 m/s^2
(close to the value in Thailand; see the box on scipy.constants.g in Chapter 2).
Reaction-time scatter of the stopwatch: standard deviation 0.15 s per trial.

Choice of SEED (stated openly, see the seed discussion in Chapter 2): with
SEED = 2026 this random draw gives g = 9.63 m/s^2 from the slope of T^2 vs L,
about 3 standard uncertainties from the true 9.78 m/s^2, an unusual draw that
would confuse a first chapter. SEED = 2028 gives a typical draw (g = 9.79).
This is a teaching data set, not a research result.
"""
import math
from pathlib import Path

import numpy as np

SEED = 2028
G_TRUE = 9.78                 # m/s^2
SIGMA_T10 = 0.15              # s, hand timing of 10 swings
LENGTHS = [0.20, 0.40, 0.60, 0.80, 1.00]   # m
N_TRIALS = 10

HERE = Path(__file__).resolve().parent
DATA = HERE / "data"
DATA.mkdir(exist_ok=True)
rng = np.random.default_rng(SEED)

rows = []
for L in LENGTHS:
    T_true = 2 * math.pi * math.sqrt(L / G_TRUE)
    t10 = np.round(10 * T_true + rng.normal(0, SIGMA_T10, N_TRIALS), 2)
    for k, t in enumerate(t10, start=1):
        rows.append((L, k, t))

path = DATA / "pendulum.csv"
with open(path, "w", encoding="utf-8") as f:
    f.write("# SYNTHETIC pendulum data for teaching (g_true = 9.78 m/s^2)\n")
    f.write("# L_m, trial, t10_s  (t10 = time for 10 complete swings)\n")
    for L, k, t in rows:
        f.write(f"{L:.2f},{k},{t:.2f}\n")
print(f"wrote {path.name}: {len(rows)} rows")
for L in LENGTHS:
    vals = [t for (LL, _, t) in rows if LL == L]
    print(f"L = {L:.2f} m  t10 = {[float(v) for v in vals]}")

## `02_variables_units.py` — โค้ดที่ 0.1, 0.2, 0.3

Ch.0  Variables, numeric types, scientific notation, f-strings and units.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 02_variables_units.py  จากโฟลเดอร์ของบท
__file__ = '02_variables_units.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.0  Variables, numeric types, scientific notation, f-strings and units.

Book references: sec:py-variables, code:gravity-moon, code:float-precision, code:ev-joule
"""
import math

### โค้ดที่ 0.1

In [ ]:
# --- code:gravity-moon -------------------------------------------------------
G = 6.674e-11          # gravitational constant [N m^2 / kg^2]
M_earth = 5.972e24     # mass of the Earth [kg]
M_moon = 7.342e22      # mass of the Moon [kg]
r_m = 3.844e8          # mean Earth-Moon distance [m]

F_N = G * M_earth * M_moon / r_m**2
print(F_N)                          # full precision
print(f"F = {F_N:.3e} N")           # 3 decimals in scientific notation
print(type(F_N), type(10), type("10"))

### โค้ดที่ 0.2

In [ ]:
# --- code:float-precision ----------------------------------------------------
a = 0.1 + 0.2
print(a)                            # 0.30000000000000004
print(a == 0.3)                     # False
print(math.isclose(a, 0.3))         # True: compare within a tolerance

### โค้ดที่ 0.3

In [ ]:
# --- code:ev-joule ------------------------------------------------------------
e_C = 1.602176634e-19               # elementary charge [C], exact by definition
E_eV = 13.6                         # hydrogen ionization energy [eV]
E_J = E_eV * e_C                    # 1 eV = e x 1 V
print(f"{E_eV} eV = {E_J:.4e} J")
print(f"back to eV: {E_J / e_C:.1f} eV")

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ----------------------------------------------------------
print("7 / 2 =", 7 / 2, "  7 // 2 =", 7 // 2, "  7 % 2 =", 7 % 2, "  2**10 =", 2**10)
print("E0.1: 2 + 3 * 4 ** 2 / 8 =", 2 + 3 * 4 ** 2 / 8)

## `03_collections.py` — โค้ดที่ 0.4, 0.5

Ch.0  Lists, tuples and dictionaries holding physical measurements.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 03_collections.py  จากโฟลเดอร์ของบท
__file__ = '03_collections.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.0  Lists, tuples and dictionaries holding physical measurements.

Book references: sec:py-collections, code:pendulum-list, code:particles-dict
"""

### โค้ดที่ 0.4

In [ ]:
# --- code:pendulum-list ------------------------------------------------------
# time for 10 swings [s], pendulum length L = 0.80 m, ten trials
t10_s = [17.66, 18.02, 17.76, 17.93, 17.93, 18.28, 17.97, 17.97, 17.90, 18.29]

print(len(t10_s))          # number of trials
print(t10_s[0])            # first trial (index starts at 0)
print(t10_s[-1])           # last trial
print(t10_s[2:5])          # trials 3, 4 and 5 (the end index is excluded)
t10_s.append(18.02)        # an eleventh trial
print(len(t10_s))

position_m = (0.0, 1.5, -0.3)   # tuple: a point (x, y, z) that should not change

### โค้ดที่ 0.5

In [ ]:
# --- code:particles-dict -----------------------------------------------------
particles = {
    "electron": {"mass_kg": 9.1093837e-31, "charge_C": -1.602176634e-19},
    "proton":   {"mass_kg": 1.67262192e-27, "charge_C": +1.602176634e-19},
    "muon":     {"mass_kg": 1.883531627e-28, "charge_C": -1.602176634e-19},
}
print(particles["muon"]["mass_kg"])
print(particles["muon"]["mass_kg"] / particles["electron"]["mass_kg"])

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ----------------------------------------------------------
print("[1, 2] * 2 =", [1, 2] * 2)
print("position_m =", position_m)

## `04_control_flow.py` — โค้ดที่ 0.6, 0.7, 0.8

Ch.0  Conditions and loops: spring regimes, averaging, free fall step by step.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 04_control_flow.py  จากโฟลเดอร์ของบท
__file__ = '04_control_flow.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.0  Conditions and loops: spring regimes, averaging, free fall step by step.

Book references: sec:py-control, code:spring-regime, code:mean-loop, code:euler-fall, exercise E0.5
"""
import math

### โค้ดที่ 0.6

In [ ]:
# --- code:spring-regime ------------------------------------------------------
F_lim_N = 10.0      # limit of Hooke's law for this spring [N]
F_break_N = 25.0    # the spring breaks above this force [N]

for F_N in [4.0, 10.0, 18.0, 30.0]:
    if F_N <= F_lim_N:
        regime = "Hooke (linear)"
    elif F_N <= F_break_N:
        regime = "beyond Hooke (nonlinear)"
    else:
        regime = "broken"
    print(f"F = {F_N:5.1f} N  ->  {regime}")

### โค้ดที่ 0.7

In [ ]:
# --- code:mean-loop ----------------------------------------------------------
t10_s = [17.66, 18.02, 17.76, 17.93, 17.93, 18.28, 17.97, 17.97, 17.90, 18.29]

total = 0.0
for t in t10_s:
    total = total + t / 10          # period of one swing [s]
T_mean_s = total / len(t10_s)
print(f"mean period T = {T_mean_s:.4f} s")

for k, t in enumerate(t10_s, start=1):
    if k <= 3:
        print(f"trial {k}: T = {t / 10:.3f} s")

### โค้ดที่ 0.8

In [ ]:
# --- code:euler-fall ---------------------------------------------------------
g = 9.78            # [m/s^2]
h0 = 20.0           # drop height [m]
dt = 0.01           # time step [s]

t, y, v = 0.0, h0, 0.0
while y > 0:
    y = y + v * dt          # position from the old velocity
    v = v - g * dt          # velocity from the acceleration
    t = t + dt

t_exact = math.sqrt(2 * h0 / g)
print(f"Euler : t = {t:.3f} s")
print(f"exact : t = {t_exact:.3f} s")

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ----------------------------------------------------------
for dt in [0.1, 0.01, 0.001]:
    t, y, v = 0.0, h0, 0.0
    while y > 0:
        y = y + v * dt
        v = v - g * dt
        t = t + dt
    print(f"dt = {dt:<6} t = {t:.4f} s  error = {t - t_exact:+.4f} s")

## `05_functions.py` — โค้ดที่ 0.9, 0.10

Ch.0  Functions with units in the docstring, checked against physics.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 05_functions.py  จากโฟลเดอร์ของบท
__file__ = '05_functions.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.0  Functions with units in the docstring, checked against physics.

Book references: sec:py-functions, code:g-function, code:projectile-range, exercise E0.6
"""
import math
from math import pi

### โค้ดที่ 0.9

In [ ]:
# --- code:g-function ---------------------------------------------------------
def g_from_pendulum(L_m, T_s):
    """Gravitational acceleration from a simple pendulum.

    L_m : length of the pendulum [m]
    T_s : period of one complete swing [s]
    returns g [m/s^2]  (small-amplitude formula T = 2 pi sqrt(L/g))
    """
    return 4 * pi**2 * L_m / T_s**2

print(f"g = {g_from_pendulum(0.80, 1.7971):.3f} m/s^2")

### โค้ดที่ 0.10

In [ ]:
# --- code:projectile-range ---------------------------------------------------
def projectile_range(v0_ms, theta_deg, g=9.78):
    """Horizontal range on level ground [m]; v0 in m/s, angle in DEGREES."""
    theta_rad = math.radians(theta_deg)
    return v0_ms**2 * math.sin(2 * theta_rad) / g

for angle in [30, 45, 60]:
    print(f"theta = {angle} deg  R = {projectile_range(20.0, angle):.2f} m")

wrong = 20.0**2 * math.sin(2 * 45) / 9.78     # forgot to convert to radians
print(f"without radians: R = {wrong:.2f} m")

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ----------------------------------------------------------
best = max(range(0, 91), key=lambda a: projectile_range(20.0, a))
print("angle with the largest range:", best, "deg")
print("R(30) == R(60)?", math.isclose(projectile_range(20, 30), projectile_range(20, 60)))

## `06_numpy_matplotlib.py` — โค้ดที่ 0.11, 0.12, 0.13, 0.14

Ch.0  First steps with NumPy arrays and Matplotlib figures.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 06_numpy_matplotlib.py  จากโฟลเดอร์ของบท
__file__ = '06_numpy_matplotlib.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.0  First steps with NumPy arrays and Matplotlib figures.

Book references: sec:py-numpy, code:np-stats, code:loadtxt, code:t2-vs-l, code:euler-plot, exercise E0.7
"""
import math
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = Path(__file__).resolve().parent
OUT = HERE / "outputs"
OUT.mkdir(exist_ok=True)

### โค้ดที่ 0.11

In [ ]:
# --- code:np-stats -----------------------------------------------------------
t10_s = np.array([17.66, 18.02, 17.76, 17.93, 17.93, 18.28, 17.97, 17.97, 17.90, 18.29])
T_s = t10_s / 10                      # every element divided by 10 at once
N = len(T_s)
T_mean = np.mean(T_s)
s = np.std(T_s, ddof=1)               # sample standard deviation
se = s / np.sqrt(N)                   # standard error of the mean
print(f"T = {T_mean:.4f} +/- {se:.4f} s   (s = {s:.4f} s, N = {N})")
print(f"np.std with ddof=0 gives {np.std(T_s):.4f} s")

### โค้ดที่ 0.12

In [ ]:
# --- code:loadtxt ------------------------------------------------------------
data = np.loadtxt(HERE / "data" / "pendulum.csv", delimiter=",", comments="#")
L_all, t10_all = data[:, 0], data[:, 2]
L = np.unique(L_all)                                   # the five lengths
T_mean_L = np.array([np.mean(t10_all[L_all == x]) / 10 for x in L])
print("L [m]      :", L)
print("T mean [s] :", np.round(T_mean_L, 4))

### โค้ดที่ 0.13

In [ ]:
# --- code:t2-vs-l ------------------------------------------------------------
slope, intercept = np.polyfit(L, T_mean_L**2, 1)       # T^2 = (4 pi^2 / g) L
g_fit = 4 * np.pi**2 / slope
print(f"slope = {slope:.4f} s^2/m  intercept = {intercept:.4f} s^2  g = {g_fit:.3f} m/s^2")

plt.figure(figsize=(5, 3.5))
plt.plot(L, T_mean_L**2, "o", label="measured (mean of 10 trials)")
plt.plot(L, slope * L + intercept, "-", label=f"fit: g = {g_fit:.2f} m/s$^2$")
plt.xlabel("length L [m]")
plt.ylabel("period squared T$^2$ [s$^2$]")
plt.legend()
plt.tight_layout()
plt.savefig(OUT / "ch00_t2_vs_L.png", dpi=150)

### โค้ดที่ 0.14

In [ ]:
# --- code:euler-plot ---------------------------------------------------------
g, h0, dt = 9.78, 20.0, 0.1
t_list, y_list = [0.0], [h0]
t, y, v = 0.0, h0, 0.0
while y > 0:
    y, v, t = y + v * dt, v - g * dt, t + dt
    t_list.append(t); y_list.append(y)

t_exact = np.linspace(0, math.sqrt(2 * h0 / g), 200)
plt.figure(figsize=(5, 3.5))
plt.plot(t_exact, h0 - 0.5 * g * t_exact**2, "-", label="exact")
plt.plot(t_list, y_list, "o--", ms=4, label=f"Euler, dt = {dt} s")
plt.xlabel("time t [s]")
plt.ylabel("height y [m]")
plt.legend()
plt.tight_layout()
plt.savefig(OUT / "ch00_euler_fall.png", dpi=150)

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ----------------------------------------------------------
(k, b), cov = np.polyfit(L, T_mean_L**2, 1, cov=True)
sk, sb = np.sqrt(np.diag(cov))
print(f"with uncertainties: slope = {k:.3f} +/- {sk:.3f} s^2/m, intercept = {b:.4f} +/- {sb:.4f} s^2, "
      f"g = {4 * np.pi**2 / k:.3f} +/- {4 * np.pi**2 / k * sk / k:.3f} m/s^2")
g_each = 4 * np.pi**2 * L / T_mean_L**2
print("g from each length:", np.round(g_each, 3), " mean =", round(float(np.mean(g_each)), 3))
print("list * 2 :", [1.0, 2.0] * 2, "   array * 2 :", np.array([1.0, 2.0]) * 2)

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `07_errors.py` — โค้ดที่ 0.15

Ch.0  Reading error messages (tracebacks) of the most common mistakes.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 07_errors.py  จากโฟลเดอร์ของบท
__file__ = '07_errors.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.0  Reading error messages (tracebacks) of the most common mistakes.

Book references: sec:py-errors, code:traceback, exercise E0.8
Each mistake is run inside try/except so that this script keeps going;
in a notebook you would see the traceback directly.
"""

### โค้ดที่ 0.15

In [ ]:
# --- code:traceback ----------------------------------------------------------
import traceback

def kinetic_energy(m_kg, v_ms):
    return 0.5 * m_kg * v_ms**2

speeds = ["3.0", "4.5"]            # read from a text file: these are strings
try:
    KE = kinetic_energy(2.0, speeds[0])
except TypeError:
    traceback.print_exc(limit=2)

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ----------------------------------------------------------
print("fixed:", kinetic_energy(2.0, float(speeds[0])), "J")
cases = {
    "NameError": "print(velocity)",
    "TypeError": "'g = ' + 9.78",
    "IndexError": "[1, 2, 3][3]",
    "ZeroDivisionError": "1 / 0",
    "KeyError": "{'mass_kg': 1.0}['mass']",
}
for name, code in cases.items():
    try:
        eval(code)
    except Exception as e:
        print(f"{name:18s} <- {code:28s} : {type(e).__name__}: {e}")

# แนวคำตอบของแบบฝึกหัดที่ต้องรันโค้ด
ควรลองทำเองก่อน แล้วจึงรันเซลล์เหล่านี้เพื่อเทียบคำตอบ

## `exercises/E0_2_photon_energy.py`

E0.2  Energy of a green laser photon (532 nm) in joules and electronvolts.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E0_2_photon_energy.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E0_2_photon_energy.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E0.2  Energy of a green laser photon (532 nm) in joules and electronvolts."""
h = 6.62607015e-34        # Planck constant [J s], exact
c = 299792458.0           # speed of light [m/s], exact
e = 1.602176634e-19       # elementary charge [C], exact
lam_m = 532e-9            # wavelength [m]
E_J = h * c / lam_m
print(f"E = {E_J:.3g} J = {E_J / e:.3g} eV")

## `exercises/E0_3_charge_to_mass.py`

E0.3  Charge-to-mass ratio of three particles stored in a dictionary.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E0_3_charge_to_mass.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E0_3_charge_to_mass.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E0.3  Charge-to-mass ratio of three particles stored in a dictionary."""
particles = {
    "electron": {"mass_kg": 9.1093837e-31, "charge_C": -1.602176634e-19},
    "proton":   {"mass_kg": 1.67262192e-27, "charge_C": +1.602176634e-19},
    "muon":     {"mass_kg": 1.883531627e-28, "charge_C": -1.602176634e-19},
}
for name, p in particles.items():
    print(f"{name:9s} q/m = {p['charge_C'] / p['mass_kg']:+.4e} C/kg")

## `exercises/E0_4_loop_mean_max.py`

E0.4  Mean and maximum of the periods with a loop, checked against sum() and max().

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E0_4_loop_mean_max.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E0_4_loop_mean_max.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E0.4  Mean and maximum of the periods with a loop, checked against sum() and max()."""
t10_s = [17.66, 18.02, 17.76, 17.93, 17.93, 18.28, 17.97, 17.97, 17.90, 18.29]
T = [t / 10 for t in t10_s]
total, largest = 0.0, T[0]
for x in T:
    total += x
    if x > largest:
        largest = x
mean = total / len(T)
print(f"loop    : mean = {mean:.4f} s  max = {largest:.3f} s")
print(f"builtin : mean = {sum(T) / len(T):.4f} s  max = {max(T):.3f} s")

## `exercises/E0_5_euler_dt.py`

E0.5  Error of the Euler free-fall time for three time steps.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E0_5_euler_dt.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E0_5_euler_dt.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E0.5  Error of the Euler free-fall time for three time steps."""
import math
g, h0 = 9.78, 20.0
t_exact = math.sqrt(2 * h0 / g)
for dt in [0.1, 0.01, 0.001]:
    t, y, v = 0.0, h0, 0.0
    while y > 0:
        y = y + v * dt
        v = v - g * dt
        t = t + dt
    print(f"dt = {dt:<6} t = {t:.4f} s  error = {t - t_exact:+.4f} s  error/dt = {(t - t_exact) / dt:.2f}")

## `exercises/E0_6_range_symmetry.py`

E0.6  The range is largest at 45 degrees and equal for complementary angles.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E0_6_range_symmetry.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E0_6_range_symmetry.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E0.6  The range is largest at 45 degrees and equal for complementary angles."""
import math

def projectile_range(v0_ms, theta_deg, g=9.78):
    """Horizontal range on level ground [m]; v0 in m/s, angle in degrees."""
    return v0_ms**2 * math.sin(2 * math.radians(theta_deg)) / g

best = max(range(0, 91), key=lambda a: projectile_range(20.0, a))
print("largest range at", best, "deg:", round(projectile_range(20.0, best), 2), "m")
for a in [10, 20, 30, 40]:
    print(a, 90 - a, math.isclose(projectile_range(20.0, a), projectile_range(20.0, 90 - a)))

## `exercises/E0_7_pendulum_g.py`

E0.7  g from the slope of T^2 versus L, compared with g from each length.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E0_7_pendulum_g.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E0_7_pendulum_g.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E0.7  g from the slope of T^2 versus L, compared with g from each length."""
from pathlib import Path
import numpy as np
HERE = Path(__file__).resolve().parent.parent
data = np.loadtxt(HERE / "data" / "pendulum.csv", delimiter=",", comments="#")
L_all, t10_all = data[:, 0], data[:, 2]
L = np.unique(L_all)
T = np.array([np.mean(t10_all[L_all == x]) / 10 for x in L])
slope, intercept = np.polyfit(L, T**2, 1)
print(f"slope fit     : g = {4 * np.pi**2 / slope:.3f} m/s^2 (intercept {intercept:+.4f} s^2)")
g_each = 4 * np.pi**2 * L / T**2
print(f"each length   : {np.round(g_each, 3)}")
print(f"mean of five  : g = {np.mean(g_each):.3f} +/- {np.std(g_each, ddof=1) / np.sqrt(len(g_each)):.3f} m/s^2")